# Quantum Simulation of H₂O₂ O–O Bond Dissociation with VQE

**Complete reproducible research notebook**

This notebook is the end-to-end presentation layer for the repository. It connects molecular geometry, classical references, active-space Hamiltonians, fermion-to-qubit mappings, VQE, dissociation-surface analysis, and resource estimation.

**Target:** CAS(2,2) H₂O₂ in STO-3G.  **Chemical accuracy:** 1.6 mHa.

> This is a methodological benchmark. It does not claim quantum advantage or hardware performance.

## 1. Workflow

H₂O₂ geometry grid → RHF/CAS(2,2)/FCI → Qiskit Nature Hamiltonian → JW/BK/parity mappings → symmetry reduction → UCCSD/HEA VQE → PES → resource analysis → conclusions.

In [ ]:
from pathlib import Path
import sys, importlib.metadata as md
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.geometry import build_dissociation_grid, R_OO_GRID
from src.classical_reference import run_classical_references
from src.hamiltonian import build_active_space_problem, validate_hamiltonians
from src.mapping import compare_mappings
from src.resource_estimation import compare_uccsd_resources
from src.vqe import benchmark_equilibrium, run_vqe_single_point
from qiskit_algorithms.optimizers import COBYLA
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD
from qiskit_nature.second_q.mappers import JordanWignerMapper

print('Repository:', ROOT)
for p in ['numpy','scipy','pandas','matplotlib','qiskit','qiskit-nature','qiskit-algorithms','pyscf']:
    try: print(f'{p:20s}', md.version(p))
    except md.PackageNotFoundError: print(f'{p:20s}', 'NOT INSTALLED')

## 2. Molecular geometry

The O–O distance is scanned at seven points while the O–H bond length and angular parameters remain fixed. Geometry files are written to `geometries/xyz/`.

In [ ]:
catalog = build_dissociation_grid(ROOT / 'geometries')
geometry_df = pd.DataFrame([{'geom_id':k, **v} for k,v in catalog.items()])
geometry_df[['geom_id','r_oo_angstrom','charge','spin']]


## 3. Classical reference calculations

RHF provides the mean-field reference, CAS(2,2) is the direct target for the reduced quantum problem, and full-space FCI provides an exact STO-3G reference.

In [ ]:
refs = run_classical_references(ROOT / 'geometries/h2o2_grid_catalog.json', ROOT / 'results/tables')
refs


## 4. Hamiltonian validation

Qiskit Nature constructs the second-quantized active-space Hamiltonian. Exact diagonalization of its qubit representation is used to verify agreement with the independent CAS(2,2) reference before any variational optimization.

In [ ]:
validation = pd.DataFrame(validate_hamiltonians(ROOT / 'geometries/h2o2_grid_catalog.json', ROOT / 'results/tables/classical_reference_energies.json'))
validation


## 5. Mapping and symmetry reduction

At R(O–O)=1.45 Å we compare Jordan–Wigner, Bravyi–Kitaev, standard parity, and particle-number-reduced parity.

The reduced parity representation demonstrates how conserved symmetries can shrink the computational problem from four to two qubits for this CAS(2,2) system.

In [ ]:
eq_id = min(catalog, key=lambda k: abs(catalog[k]['r_oo_angstrom']-1.45))
eq = catalog[eq_id]
cas_eq = float(refs.loc[refs.geom_id==eq_id,'e_cas22_hartree'].iloc[0])
problem = build_active_space_problem(eq['pyscf_atom_string'])
mapping_df = pd.DataFrame(compare_mappings(problem, cas_eq))
mapping_df


In [ ]:
mapping_df[['mapping','num_qubits','num_pauli_terms','avg_pauli_weight','max_pauli_weight']]


## 6. Equilibrium VQE benchmark

UCCSD is compared with a two-qubit hardware-efficient EfficientSU2 circuit. The ideal statevector estimator establishes the baseline without hardware noise.

In [ ]:
vqe_eq = pd.DataFrame(benchmark_equilibrium(problem, cas_eq))
vqe_eq.assign(chemical_accuracy=vqe_eq.error_mha <= 1.6)


## 7. Circuit-resource analysis

We quantify qubits, Hamiltonian Pauli terms, total gates, CNOT gates, and circuit depth for UCCSD under standard JW and reduced parity.

In [ ]:
resources = pd.DataFrame(compare_uccsd_resources(problem))
resources


## 8. Full dissociation potential-energy surface

The following cell runs the same UCCSD/Jordan–Wigner VQE workflow at every geometry. No energy values are hard-coded.

In [ ]:
ref = refs.set_index('geom_id')
rows=[]
for gid,data in catalog.items():
    p=build_active_space_problem(data['pyscf_atom_string'])
    fop=p.hamiltonian.second_q_op(); offset=sum(p.hamiltonian.constants.values())
    mapper=JordanWignerMapper(); qop=mapper.map(fop)
    hf=HartreeFock(p.num_spatial_orbitals,p.num_particles,mapper)
    ansatz=UCCSD(p.num_spatial_orbitals,p.num_particles,mapper,initial_state=hf)
    result,history=run_vqe_single_point(ansatz,qop,COBYLA(maxiter=200),[0.0]*ansatz.num_parameters)
    e=float(result.eigenvalue.real+offset); cas=float(ref.loc[gid,'e_cas22_hartree'])
    rows.append({'geom_id':gid,'r_oo_angstrom':data['r_oo_angstrom'],'e_rhf_hartree':float(ref.loc[gid,'e_rhf_hartree']),'e_cas22_hartree':cas,'e_vqe_uccsd_hartree':e,'e_fci_full_hartree':float(ref.loc[gid,'e_fci_full_hartree']),'vqe_error_mha':abs(e-cas)*1000,'vqe_iterations':len(history),'chemical_accuracy':abs(e-cas)<=0.0016})
pes=pd.DataFrame(rows).sort_values('r_oo_angstrom')
pes.to_csv(ROOT/'results/tables/vqe_pes_benchmark.csv',index=False)
pes


In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
ax.plot(pes.r_oo_angstrom,pes.e_rhf_hartree,'--o',label='RHF')
ax.plot(pes.r_oo_angstrom,pes.e_cas22_hartree,'-s',label='CAS(2,2)')
ax.plot(pes.r_oo_angstrom,pes.e_vqe_uccsd_hartree,'--^',label='VQE UCCSD')
ax.plot(pes.r_oo_angstrom,pes.e_fci_full_hartree,':',label='Full FCI')
ax.set(xlabel='O–O distance R (Å)',ylabel='Energy (Hartree)',title='H₂O₂ O–O Dissociation PES'); ax.legend(); fig.tight_layout(); plt.show()


In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
ax.axhline(1.6,linestyle='--',label='Chemical accuracy (1.6 mHa)')
ax.plot(pes.r_oo_angstrom,pes.vqe_error_mha,'o-',label='UCCSD VQE error')
ax.set(xlabel='O–O distance R (Å)',ylabel='Absolute error (mHa)',title='VQE Accuracy Along Dissociation'); ax.legend(); fig.tight_layout(); plt.show()


## 9. Shot noise: interpretation

The current `src/noise_analysis.py` deliberately models shot noise by perturbing an exact-estimator VQE energy with a Gaussian term proportional to 1/√Nshots. This is a **statistical uncertainty model**, not a true finite-shot Pauli-measurement VQE. It must not be presented as a hardware result.

In [ ]:
from src.noise_analysis import simulate_shot_noise_vqe
print('Noise model available:', simulate_shot_noise_vqe.__name__)
print('For a hardware-grade study, replace this model with sampled Pauli-term estimation and noisy circuits.')


# 10. Results summary

The final benchmark should report: (1) whether the mapped Hamiltonian reproduces CAS, (2) qubit/Pauli-term reduction, (3) equilibrium VQE error, (4) VQE error across the PES, and (5) circuit-resource savings.

A successful run can be summarized programmatically rather than manually entering numbers.

In [ ]:
summary={
 'equilibrium_R_OO_A':float(eq['r_oo_angstrom']),
 'grid_A':R_OO_GRID,
 'min_VQE_error_mHa':float(pes.vqe_error_mha.min()),
 'max_VQE_error_mHa':float(pes.vqe_error_mha.max()),
 'all_points_chemical_accuracy':bool(pes.chemical_accuracy.all()),
 'mapping_table':mapping_df.to_dict('records'),
 'resource_table':resources.to_dict('records')
}
summary


# 11. Scientific interpretation

This study demonstrates the complete mechanics of a small quantum chemistry calculation. The central lesson is that the quantum algorithm is only one component: active-space definition controls the physical approximation, symmetry reduction controls the qubit requirement, mapping controls Hamiltonian structure, ansatz design controls expressibility and circuit cost, and the dissociation scan tests whether a method remains reliable away from equilibrium.

The project therefore serves as a compact benchmark for learning how computational chemistry, quantum algorithms, and reproducible scientific programming fit together.

# 12. Limitations and next steps

- STO-3G is a minimal basis.
- CAS(2,2) is intentionally small.
- The baseline uses an ideal estimator rather than hardware.
- The current noise section is a statistical model.
- No quantum advantage is claimed.

**Next version:** add larger active spaces, sampled Pauli measurements, explicit noisy Aer simulations, optimizer comparisons, convergence diagnostics, spin-state treatment during bond breaking, and real-backend execution.

## Reproducibility

From the repository root:

```bash
conda env create -f environment.yml
conda activate h2o2-vqe
python -m pytest -q
python run_study.py
```

The modular script produces the same classes of artifacts used by this notebook: tables under `results/tables/`, figures under `results/figures/`, and a machine-readable study summary.